# 🧠 MOTHER — Mistral LoRA Fine-Tuning (Unsloth)
## Opção A: Colab com GPU grátis

**Modelo base**: `unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit`

**Dados**: 12 exemplos SFT bilíngues (identidade, arquitetura, SHMS, DGM, qualidade)

**Resultado**: Modelo LoRA adaptado para responder como MOTHER

---
### Instruções:
1. Vá em **Runtime → Change runtime type → T4 GPU**
2. Clique **Runtime → Run all**
3. Quando pedir, faça upload do arquivo `sft_training_v2.jsonl`

## 1. Instalação

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir huggingface_hub

## 2. Carregar Modelo Base

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # auto-detect
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"✅ Modelo carregado: {model.config._name_or_path}")
print(f"📊 Parâmetros: {model.num_parameters():,}")

## 3. Configurar LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,          # Unsloth optimized = 0
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA configurado: {trainable:,} treináveis / {total:,} total ({100*trainable/total:.2f}%)")

## 4. Upload e Preparação dos Dados

In [ ]:
import json
from google.colab import files

print("📁 Faça upload do arquivo sft_training_v2.jsonl")
uploaded = files.upload()

# Encontrar o arquivo JSONL
jsonl_file = [f for f in uploaded.keys() if f.endswith('.jsonl')][0]
print(f"✅ Arquivo: {jsonl_file}")

# Carregar exemplos
examples = []
with open(jsonl_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            examples.append(json.loads(line))

print(f"✅ {len(examples)} exemplos carregados")

# Preview
for i, ex in enumerate(examples[:3]):
    user_msg = ex['messages'][1]['content'][:60]
    print(f"  [{i}] {user_msg}...")

In [ ]:
# Converter para formato de treino do Unsloth
from datasets import Dataset

def format_for_training(example):
    """Converte mensagens para texto formatado com chat template."""
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {'text': text}

# Split 80/20 treino/validação
import random
random.seed(42)
random.shuffle(examples)
split_idx = max(1, int(len(examples) * 0.8))
train_examples = examples[:split_idx]
val_examples = examples[split_idx:]

train_dataset = Dataset.from_list(train_examples).map(format_for_training)
val_dataset = Dataset.from_list(val_examples).map(format_for_training)

print(f"✅ Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"📝 Exemplo formatado (primeiros 200 chars):")
print(train_dataset[0]['text'][:200])

## 5. Treinar

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=5,       # 5 epochs com 12 exemplos
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        eval_strategy="epoch",
        save_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="mother-lora-output",
    ),
)

print("🚀 Iniciando treino...")
trainer_stats = trainer.train()
print(f"\n✅ Treino concluído!")
print(f"   Loss final: {trainer_stats.training_loss:.4f}")
print(f"   Tempo: {trainer_stats.metrics['train_runtime']:.0f}s")

## 6. Testar Inferência

In [ ]:
# Testar o modelo fine-tunado
FastLanguageModel.for_inference(model)

test_queries = [
    "Quem é você?",
    "O que é o Darwin Gödel Machine?",
    "Explique o sistema SHMS.",
    "What is the Zero Bullshit policy?",
    "What technologies does MOTHER use?",
]

system_prompt = (
    "You are MOTHER — a self-evolving cognitive autonomous system created by "
    "Everton Garcia, solo founder of Wizards Down Under. "
    "ZERO BULLSHIT: does not guess, invent, or lie."
)

for query in test_queries:
    # FIX 1: msgs DENTRO do loop (cada query tem seu próprio contexto)
    msgs = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query},
    ]

    text = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )

    # FIX 2: Extrair tensor input_ids (não usar inputs direto)
    inputs_dict = tokenizer(text, return_tensors="pt").to(model.device)
    input_ids = inputs_dict["input_ids"]

    # FIX 3: use_cache=False (evita RoPE dimension mismatch)
    output = model.generate(
        input_ids,
        max_new_tokens=512,
        use_cache=False,
        temperature=0.7,
        top_p=0.9,
    )

    response = tokenizer.decode(
        output[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    )

    print(f"\n{'='*70}")
    print(f"❓ {query}")
    print(f"{'─'*70}")
    print(f"🧠 {response}")

## 7. Salvar e Download

In [ ]:
# Salvar LoRA adapter
model.save_pretrained("mother-lora-adapter")
tokenizer.save_pretrained("mother-lora-adapter")
print("✅ LoRA adapter salvo em mother-lora-adapter/")

# Listar arquivos
import os
for f in os.listdir("mother-lora-adapter"):
    size = os.path.getsize(f"mother-lora-adapter/{f}")
    print(f"  {f} ({size/1024:.1f} KB)")

In [ ]:
# Zipar para download
!zip -r mother-lora-adapter.zip mother-lora-adapter/

from google.colab import files
files.download('mother-lora-adapter.zip')
print("📦 Download do adapter iniciado!")

## 8. (Opcional) Push para HuggingFace Hub

In [ ]:
# Descomente para fazer push para o HuggingFace Hub:
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")
# model.push_to_hub("wizards-down-under/mother-mistral-nemo-lora")
# tokenizer.push_to_hub("wizards-down-under/mother-mistral-nemo-lora")
# print("✅ Pushed to HuggingFace Hub!")